# ATH operational-v2 auth-to-execution evaluation on Colab

Run this notebook now to complete the live-model part of operational step 3. It compares the deterministic operational-v2 pipeline with `qwen3.5:4b` on three development scenarios and six held-out scenarios (two repeats). The Windows baseline is already recorded, but this notebook rebuilds both arms under one frozen Colab runtime so the paired comparison uses identical code, data and runtime.

Before running: **Runtime → Change runtime type → T4 GPU**. Run every cell in order. Expected time is roughly 30–90 minutes, depending on model response time and probe count. The result is a synthetic mechanism-level pilot, not a production accuracy estimate.

This does **not** complete M19b, which needs hosted API credit and human review. It also does not replace the older `ath_d1_model_comparison_colab.ipynb`, which asks a separate D1-v3 model-size question.

## 0. Configuration
The commit check is an ancestor check: newer documentation-only commits are allowed, while a checkout older than the operational-v2 implementation is refused. Ollama is pinned because its version is part of the experiment freeze.


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/shayb1187-a11y/agentic-threat-hunter.git"
BRANCH = "m14-real-data-validation"
MINIMUM_COMMIT = "f48379cd173d1587bc54f820ccb0885de2636b43"
MODEL = "qwen3.5:4b"
OLLAMA_VERSION = "0.34.1"
REPO = Path("/content/agentic-threat-hunter")
OUTPUT = Path("/content/ath-auth-execution-colab")
DEV = OUTPUT / "dev"
HELDOUT = OUTPUT / "heldout"
print({"model": MODEL, "output": str(OUTPUT)})

## 1. Verify the GPU
A T4 is sufficient for the 4B model. Stop and change the runtime if `nvidia-smi` is unavailable.


In [ ]:
import shutil, subprocess

assert shutil.which("nvidia-smi"), "No NVIDIA GPU: Runtime → Change runtime type → T4 GPU"
subprocess.run(["nvidia-smi"], check=True)

## 2. Clone the public branch and install ATH


In [ ]:
import os, sys

git_env = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
if not (REPO / ".git").exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO)], check=True, env=git_env)
else:
    subprocess.run(["git", "fetch", "origin", BRANCH], cwd=REPO, check=True, env=git_env)
    subprocess.run(["git", "checkout", BRANCH], cwd=REPO, check=True, env=git_env)
    subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=REPO, check=True, env=git_env)
subprocess.run(["git", "merge-base", "--is-ancestor", MINIMUM_COMMIT, "HEAD"], cwd=REPO, check=True)
subprocess.run(["git", "diff", "--quiet"], cwd=REPO, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], cwd=REPO, check=True)
os.chdir(REPO)
print("commit", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
print("python", sys.version.split()[0])

## 3. Install Ollama, start one inference slot, and pull the frozen model


In [ ]:
import json, time, urllib.request

def daemon_up():
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/version", timeout=2)
        return True
    except Exception:
        return False

if not shutil.which("ollama"):
    install = f"curl -fsSL https://ollama.com/install.sh | OLLAMA_VERSION={OLLAMA_VERSION} sh"
    subprocess.run(install, shell=True, check=True)
if not daemon_up():
    log = open("/content/ollama-auth-execution.log", "ab")
    daemon_env = {**os.environ, "OLLAMA_NUM_PARALLEL": "1", "OLLAMA_KEEP_ALIVE": "-1"}
    subprocess.Popen(["ollama", "serve"], stdout=log, stderr=subprocess.STDOUT, start_new_session=True, env=daemon_env)
    for _ in range(120):
        if daemon_up(): break
        time.sleep(1)
    else: raise RuntimeError("Ollama did not start; inspect /content/ollama-auth-execution.log")
version = json.load(urllib.request.urlopen("http://127.0.0.1:11434/api/version"))["version"]
assert version == OLLAMA_VERSION, (version, OLLAMA_VERSION)
subprocess.run(["ollama", "pull", MODEL], check=True)
print("Ollama", version, "with", MODEL)

## 4. Optional: restore a checkpoint zip
Skip this on the first session. After a disconnect, upload the zip downloaded by this notebook. Files are extracted only under the output directory. Existing raw rows are never overwritten by the evaluator and are validated before reuse. A different source tree, Python/pandas runtime, model digest, or Ollama version requires a new output directory.


In [ ]:
# OPTIONAL: uncomment in a resumed session.
# from google.colab import files
# import io, zipfile
# uploaded = files.upload()
# for name, blob in uploaded.items():
#     if not name.endswith('.zip'): continue
#     with zipfile.ZipFile(io.BytesIO(blob)) as archive:
#         for member in archive.infolist():
#             target = (Path('/content') / member.filename).resolve()
#             assert Path('/content') in target.parents, member.filename
#         archive.extractall('/content')
# print('restored', sorted(uploaded))

## 5. Freeze development and held-out protocols before inference
Both freezes are created before any model call. Do not edit source, configuration, prompts, cases, repeats, or decision criteria after this cell. If a freeze already exists, this cell validates it through `summarise` rather than replacing it.


In [ ]:
def ath(*args, allow=(0,)):
    command = [sys.executable, "-m", "ath.evaluation.auth_execution", *map(str, args)]
    print("+", " ".join(command), flush=True)
    result = subprocess.run(command, cwd=REPO)
    if result.returncode not in allow:
        raise RuntimeError(f"command exited {result.returncode}")
    return result.returncode

OUTPUT.mkdir(parents=True, exist_ok=True)
for path, split, repeats in ((DEV, "dev", 1), (HELDOUT, "heldout", 2)):
    if not (path / "FREEZE.json").exists():
        ath("freeze", "--out", path, "--split", split, "--repeats", repeats, "--model", MODEL)
    else:
        ath("summarise", "--out", path)
print("Both experiment identities are frozen.")

## 6. Run both deterministic baselines in this runtime
These rows are fast and provide the paired Colab baseline.


In [ ]:
ath("run", "--out", DEV, "--arm", "deterministic")
ath("run", "--out", HELDOUT, "--arm", "deterministic")

## 7. Run the three development model rows
This is a pipeline and compatibility check. Do not change the experiment after reading it.


In [ ]:
ath("run", "--out", DEV, "--arm", "d1", allow=(0, 3))
summary = json.loads((DEV / "SUMMARY.json").read_text())
print(json.dumps({"complete": summary["complete_comparison"], "arms": summary["arms"], "conclusion": summary["conclusion"]}, indent=2))
assert summary["complete_comparison"], "Development run is incomplete. Inspect missing_rows and the last command output before continuing."

## 8. Download a development checkpoint


In [ ]:
import zipfile
from google.colab import files

def export_results(name):
    target = Path('/content') / name
    if target.exists(): target.unlink()
    with zipfile.ZipFile(target, 'w', zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(OUTPUT.rglob('*')):
            if path.is_file(): archive.write(path, path.relative_to('/content'))
    print(target, target.stat().st_size, 'bytes')
    files.download(str(target))

export_results('ath_auth_execution_dev_checkpoint.zip')

## 9. Run the frozen held-out model comparison
This produces 12 D1 rows: six held-out scenarios × two repeats. Failures and abstentions stay in the denominator. Do not tune or delete rows after inspecting outcomes. Rerunning validates and skips completed rows.


In [ ]:
ath("run", "--out", HELDOUT, "--arm", "d1", allow=(0, 3))
ath("summarise", "--out", HELDOUT)
summary = json.loads((HELDOUT / "SUMMARY.json").read_text())
print(json.dumps(summary, indent=2))
assert summary["complete_comparison"], "Held-out comparison is incomplete; save the checkpoint and resume without changing the freeze."

## 10. Export the complete evidence bundle
Keep this zip. It contains immutable freezes, raw row JSON, per-row Markdown reports and derived summaries for both splits. To add it to the repository later, review it first and copy the two directories under a clearly named Colab-results path; do not overwrite the checked-in Windows baseline.


In [ ]:
export_results('ath_auth_execution_colab_complete.zip')

## What this measures—and what it does not

The summary measures decision accuracy, false malicious/benign dispositions, missing-data abstention, useful evidence retrieved/cited, typed parent-child recovery, rejected claims, invalid accepted predicates, latency and reported tokens. Local Ollama API cost is zero. Hardware/energy cost, analyst time savings and free-text semantic correctness remain unmeasured. A positive result is still only a small synthetic pilot.

After this notebook, the optional older `ath_d1_model_comparison_colab.ipynb` can compare 4B and 9B models under D1-v3. M19b remains separate because its missing rows require hosted API credit, and its usefulness question requires the blinded human review.